# Test the Deployed Booking Agent

Invokes the agent already running on **Amazon Bedrock AgentCore Runtime** and checks the
behaviours this demo is meant to prove. Deploy first with
[`deploy_agentcore.ipynb`](./deploy_agentcore.ipynb) — this notebook only *reads* and *invokes*,
it creates nothing.

| Test | What it proves |
|------|----------------|
| 1. Graph aggregation | Exact counts from AuraDB instead of a guess |
| 2. Hotel detail | Graph traversal returns real amenities |
| 3. Out-of-domain | Honest "no results" instead of fabrication |
| 4. Tool routing | The Gateway picks DynamoDB tools, not the graph |
| 5. Steering | Oversized party is split, not rejected |
| 6. Hard guardrail | Confirmation without payment is blocked |
| 7. Multi-turn | Session memory across calls |

**Prerequisites:** the deployment finished, AWS credentials for that account, and
(for tests 1-3) AuraDB populated — see [Step A in the README](./README.md#neo4j-setup-self-paced).

In [ ]:
import json
import os
import uuid

import boto3

try:
    from dotenv import load_dotenv
    load_dotenv()
except ModuleNotFoundError:
    pass

REGION = os.environ.get("AWS_REGION", "us-east-1")
RUNTIME_NAME = os.environ.get("RUNTIME_NAME", "HotelBookingAgent")

control = boto3.client("bedrock-agentcore-control", region_name=REGION)
data = boto3.client("bedrock-agentcore", region_name=REGION)

# Discover the runtime ARN instead of hard-coding it: the ARN embeds the AWS
# account ID, and .bedrock_agentcore.yaml (which also holds it) is git-ignored.
RUNTIME_ARN = os.environ.get("RUNTIME_ARN", "")
if not RUNTIME_ARN:
    matches = [
        r for r in control.list_agent_runtimes().get("agentRuntimes", [])
        if r.get("agentRuntimeName") == RUNTIME_NAME
    ]
    if not matches:
        raise RuntimeError(
            f"No runtime named {RUNTIME_NAME!r} in {REGION}. "
            "Run deploy_agentcore.ipynb first, or set RUNTIME_ARN."
        )
    RUNTIME_ARN = matches[0]["agentRuntimeArn"]
    status = matches[0].get("status")
    if status != "READY":
        print(f"WARNING: runtime status is {status}, not READY — calls may fail.")

print(f"Region:  {REGION}")
print(f"Runtime: {RUNTIME_NAME} (ready)")

In [ ]:
def ask(prompt, session_id=None, show=True):
    """Invoke the deployed agent once. Reuse session_id to continue a conversation."""
    session_id = session_id or str(uuid.uuid4())
    response = data.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )
    body = json.loads(response["response"].read().decode("utf-8"))
    answer = body.get("response", "")
    tools = body.get("tools_used", [])
    if show:
        print(f"Q: {prompt}")
        print(f"   tools: {tools or '(none)'}")
        print(f"   {answer}\n")
    return answer, tools, session_id

---
## Tests 1-3: Graph-RAG against AuraDB

These hit the `query_knowledge_graph` Lambda, which runs Cypher against Neo4j AuraDB. The point is
that counts are **computed**, not estimated — and that an empty result is reported honestly.

A verified full build holds 296 hotels (entity resolution merges same-name hotels across
documents), so the exact numbers depend on how much of the graph you loaded.

In [ ]:
# TEST 1 — exact aggregation over the whole graph
ask("How many hotels are in the knowledge graph? Use the knowledge graph.")
ask("Using the knowledge graph, how many hotels have a swimming pool?")

In [ ]:
# TEST 2 — graph traversal for one hotel's amenities
ask("What amenities does Adobe Plaza Inn have? Use the knowledge graph.")

In [ ]:
# TEST 3 — out-of-domain: the honest answer is "none"
ask("Using the knowledge graph, tell me about hotels in Antarctica.")

---
## Test 4: Tool routing

The Gateway routes semantically. A booking search must land on the DynamoDB-backed
`search_available_hotels`, not on the graph tool.

Note the two stores hold different data on purpose: DynamoDB carries the seeded booking inventory,
AuraDB carries the 300-document knowledge graph. Different counts between them are expected.

In [ ]:
# TEST 4 — should call search_available_hotels (DynamoDB), not the graph
ask("Search for hotels in Lisbon")
ask("Find hotels under $100 per night")

---
## Test 5: Steering (soft rules self-correct)

Steering rules live in the `workshop-SteeringRules` DynamoDB table. A party larger than the
per-room limit should be **split across rooms**, not refused: the agent receives a STEER
instruction and fixes its own call.

In [ ]:
# TEST 5 — 15 guests exceeds the 10-per-room rule; expect a split, not a refusal
ask("Book AnyCompany Paris Central for 15 guests from 2026-06-01 to 2026-06-05 for Ana Diaz")

---
## Test 6: Hard guardrail (cannot be bypassed)

`BookingGuardrailsHook` in `booking_agent.py` enforces payment-before-confirmation and the
48-hour cancellation window in code, as a `BeforeToolCallEvent` hook. Unlike a prompt rule, the
model cannot talk its way past it.

In [ ]:
# TEST 6 — confirming an unpaid/unknown booking must be blocked
ask("Confirm booking BK-FAKE123 without payment")

---
## Test 7: Session memory

Passing the same `runtimeSessionId` continues the conversation, so the follow-up resolves
"that hotel" from the previous turn.

In [ ]:
# TEST 7 — follow-up in the same session
_, _, sid = ask("Find me a 4-star hotel in London")
ask("What is the nightly rate for that hotel?", session_id=sid)

---
## Your turn

Edit the prompt below. Ask for an aggregation ("average nightly rate across all rooms") to see the
graph compute it, or something absent from the data to see it decline instead of inventing.

In [ ]:
ask("Using the knowledge graph, what is the average guest rating of hotels in Las Vegas, "
    "and how many hotels did you use?")

---
**Cleanup:** `python cleanup.py` removes every resource created by the deployment.

**Next:** [Demo 06 with CDK](../06-agentcore-production-demo/) for the infrastructure-as-code path.